In [3]:
import requests, csv
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from urllib.request import Request, urlopen
import re
import tqdm
import json
import tqdm
from difflib import SequenceMatcher
import undetected_chromedriver as uc
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

In [4]:
head = {
      'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.84 Safari/537.36',
      'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
      'Accept-Charset': 'ISO-8859-1,latin-1;q=0.7,*;q=0.3',
      'Accept-Encoding': 'none',
      'Accept-Language': 'fr;q=0.9,en-US,en;q=0.8',
      'Connection': 'keep-alive',
      'refere': 'https://imdb.com'}

In [5]:
def month_movie(year, month):
    
    month_dict = {'january':'01', 'february':'02', 'march':'03', 'april':'04', 'may':'05', 'june':'06', \
                'july':'07', 'august':'08', 'september':'09', 'october':'10', 'november':'11', 'december':'12'}
    
    req1 = Request(f"https://www.allocine.fr/film/agenda/mois/mois-{year}-{month_dict[month]}", headers=head)
    webpage1 = urlopen(req1).read()
    bs = BeautifulSoup(webpage1, "html.parser")
    movies = bs.findAll('a', {'class': "month-movies-link"})
    movie_list = [text.text.strip() for text in movies]

    return movie_list

In [6]:
month = month_movie(2026, 'april')

In [8]:
def get_similar_movies(driver, query: str) -> pd.DataFrame:

    query_dict = {"\xa0": "","\x9c": "oe",'\u0153':"oe"," !": "!"," ?": "?"," :": ":","’": "'","–": "-","\x92": "'","\xc2":"â",\
                  "\xe1":"a","\xf1":"n","\xf6":"o","\x96":"-","\x8c":"oe","\xae":"","ì":"i","\xfc":"u","\xdc":"U"}
    
    # dictionary of characters to change in URL
    char_dict = {" ": "%20","!": "%21", "#": "%23", "$": "%24", "&": "%26", "'": "%27", "(": "%28", ")": "%29", \
                 "*": "%2A", "+": "%2B", ",": "%2C", "/": "%2F", ":": "%3A", ";": "%3B", "=": "%3D", "?": "%3F", \
                 "@": "%40", "[": "%5B", "]": "%5D", "â": "a", "à": "a", "é": "e", "è": "e", "ê": "e", "ë": "e", \
                 "ç": "c", "ï": "i", "ô": "o", "î": "i", "’": "%27", "ó":"o", "ł":"l", "ù":"u","Î":"i","ú":"u", \
                "'":"%27","À":"a","·":".","°":"","č":"c","É":"e","œ":"oe","û":"u","Ç":"c"}
    
    query = query.strip()
    for old, new in query_dict.items():
        query = query.replace(old, new)

    modif_query = [char_dict[i] if i in char_dict.keys() else i for i in query]
    modif_query = "".join(modif_query) + "&ref_=nv_sr_sm"

          
    # 1. PAGE PRINCIPALE (Infos, Budget, Directors)
    url_main = f"https://www.imdb.com/find/?s=tt&q={modif_query}"
    driver.get(url_main)
    time.sleep(random.uniform(4, 6))
                
    bs = BeautifulSoup(driver.page_source, "html.parser")
    items = bs.select('li[class*="ipc-metadata-list-summary-item"]')[:6]
    
    results = []
    for li_tag in items:
        # Titre
        title_tag = li_tag.select_one("h3.ipc-title__text")
        if not title_tag:
            continue
        title = title_tag.get_text(strip=True)
        
        # Code IMDb
        link_tag = li_tag.select_one("a[href*='/title/tt']")
        if not link_tag:
            continue
        imdb_id = link_tag["href"].split("/title/")[1].split("/")[0]
        
        # year
        try: 
            text = li_tag.select("span")[1].text
        
            match = re.search(r"(19|20)\d{2}", text)
            year = int(match.group()) if match else 1492
        except ValueError:
            year = 1492
    
        # Similarity
        similarity = SequenceMatcher(None, query.lower(), title.lower()).ratio()
        
        if similarity >= 0.6 and year > 2022:
            results.append({
                "imdb_id": imdb_id,
                "title": title,
                "query": query,
                "year": year,
                "similarity": round(similarity, 3)
            })

        if len(results) == 0:
            results = [{"imdb_id": "", "title": "", "query": query, "year": "", "similarity": ""}]    

    return pd.DataFrame(results)

In [21]:
options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=145)
driver.execute_cdp_cmd("Network.enable", {})
    
driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
    "headers": {
        "Accept-Language": "en-US,en;q=0.9,fr;q=0.9"
    }
})

{}

In [22]:
new_df = pd.DataFrame()

In [23]:
try:
    for movie in tqdm.tqdm(month):
        new_df = pd.concat([new_df, get_similar_movies(driver, movie)])
        
finally:
    print("Fermeture du navigateur...")
    driver.quit()

100%|████████████████████████████████████████| 123/123 [18:35<00:00,  9.07s/it]


Fermeture du navigateur...


In [24]:
new_df

,imdb_id,title,query,year,similarity
0,tt33071426,The Drama,The Drama,2026,1.0
0,tt28650488,The Super Mario Galaxy Movie,Super Mario Galaxy Le Film,2026,0.741
0,tt39814688,Compostelle,Compostelle,2026,1.0
0,,,Plus fort que moi,,
0,tt39314864,Mauvaise pioche,Mauvaise Pioche,2026,1.0
...,...,...,...,...,...
0,,,Afriques: comment ca va avec la douleur?,,
0,,,"Empty Quarter, une femme en Afrique",,
0,,,Un Homme sans l'Occident,,
0,tt40622377,Power to The People: John & Yoko Live in NYC,Power To The People: John & Yoko Live In NYC,2026,1.0


In [25]:
new_df.to_excel("new_movies2.xlsx")

In [20]:
month

['The Drama',
 'Super Mario Galaxy Le Film',
 'Compostelle',
 'Plus fort que moi',
 'Mauvaise Pioche',
 'Yellow Letters',
 'Silent Friend',
 'Derrière les palmiers',
 'Dolly',
 'Nuestra Tierra',
 'Le Goût des autres',
 'Holding Liat',
 'Hélène Trésore Transnationale',
 'Le Secret du Loup d’Éthiopie',
 'Peppa au cinéma : La famille s’agrandit !',
 'Wives',
 'Le Fleuve de la mort',
 'Wives 2, dix ans après',
 "L'Heure de la libération a sonné",
 'Wives 3, elles ont 50 ans !',
 'Cocorico 2',
 "L'Enfant Du Désert",
 'Wedding Nightmare : deuxième partie',
 'La Femme de',
 'Sauvage',
 'Romería',
 'Le Cri des gardes',
 'Le dernier pour la route',
 'L’Affaire Abdallah',
 'Les Contes du pommier',
 'Pour Klára',
 "L'Oeuvre invisible",
 'Bisons',
 'Sur le sentier',
 "BTS World Tour 'Arirang' in GOYANG : Live Viewing",
 'The World is Full of Secrets',
 'Dans la chambre du sultan',
 'An Evening Song (for three voices)',
 'La Ville et les Chiens',
 'La Randonnée',
 'Little Feet',
 'Tosca (Opéra de P